# Compression Example

To compress several estiators output in the format required by the `Observable` class in the next step of the pipeline, we need to use the `Compressor` class. This class will read several `.h5` files, merge them, and compress them into a single `xarray.DataArray` object.

In [ ]:
# Global setup
import itertools
import logging
from pathlib import Path

import lsstypes
from estimators.helpers import make_dummy_lsstypes_object

from acm.utils.logging import setup_logging

logger = logging.getLogger("Compression examples")
setup_logging()

# First, let's make a dummy lsstypes object:
tree = make_dummy_lsstypes_object()

# Now, let's create a function to generate the mock files in a temporary directory for testing the Compressor class.
def generate_mock_files(tmp_path: Path) -> None:
    """Generate mock .h5 files in a temporary directory for testing the Compressor class."""
    for i, j, k, v in itertools.product(range(2), range(2), range(1, 4), range(1, 2)):
        dir_path = tmp_path / f"I{i}/J{j}_K{k}/v{v}.0"
        dir_path.mkdir(parents=True, exist_ok=True)
        for m in range(1, 3):  # Create two files for each combination of i, j, k, v
            file_path = dir_path / f"file_M{m}.h5"
            lsstypes.write(file_path, tree)  # Write the mock lsstypes object to the .h5 file

generate_mock_files(tmp_path=Path("mock_data"))

First, we can identify the files we want to compress t
rough pattern matching. The `Compressor` class will create a `Pattern` object, that can confert a formatted string into a glob or regex pattern.

In [ ]:
from acm.estimators.compression import Pattern

root = Path("mock_data")
pattern_str = "I{i}/J{j}_K{k}/{l}/file_{m}.h5" # Let's replace the version by the l index to have a string value

pattern = Pattern(root=root, pattern=pattern_str)

print("Reconstructed glob pattern: ", pattern.to_glob())
print("Reconstructed regex pattern: ", pattern.to_regex(), "with groups: ", pattern.names)

In [ ]:
from acm.estimators.compression import Compressor

# First, the Compressor instance registers matching files trough glob pattern matching.
compressor = Compressor(root=root, pattern=pattern_str)

The compressor can then read the files in the list. We can specify indexes to ignore when reading the files. 
The data is stored in a `ObjectGroup` instance, which contains `IndexedObject` instances. The `ObjectGroup` can only contain `IndexedObject` instances with the same indexes.

In [ ]:
group = compressor.read(reader=lsstypes.read, ignore_index=["m"])

print("Names of the group indexes: ", group.names)

# We can select specific indices of the group !
subgroup = group.get(i='0', j='1', k='2')  # Selects the subgroup with i=0, j=1, k=2 - Note that the indices are strings because they were extracted from the file paths, which are strings.
print(f"Found {len(subgroup)} elements with i=0, j=1, k=2")

In the case where some indexes are ignored, there might be several files that have the same final indexes. In that case, you can use the `merge` method to merge the `IndexedObject` instances with the same indexes. The `merge` method takes a `method` argument, which must take a list of `lsstypes` objects and return a single `lsstypes` object.

In [ ]:
group = group.merge(method=lsstypes.mean)  # Merge the IndexedObject instances with the same indexes using the mean method

print(f"After merging, there are {len(group)} elements remaining in the group.")

If you need to apply transformations to the data, you can access the `lsstypes` object methods directly on the group (trough a `getattr` override). This will apply the method to all `lsstypes` objects in the group, and return a new `ObjectGroup` instance with the transformed data.

In [ ]:
group = group.select(s=(0, 50))  # Selects the s values between 0 and 50

group.objects[0].data.flatten(level=None)[0].coords() # Print the first object in the group to see its contents

The `ObjectGroup` class has some list properties. It is ordered in the order of the sorted files, which should usually be enough to keep the order of the indexes.

In the case where you want to change the order of the objects to match some nested index sorting, you can use the `sort` method, which will sort the `IndexedObject` instances in the group according to the specified order of indexes. The `sort` method takes a list of index names, and will sort the `IndexedObject` instances in the group according to the specified order of indexes.

> ***Note:** The `sort` method does not modify the index order of the `IndexedObject` instances, it only changes the order of the objects in the group. The index order is still determined by the `IndexedObject` instances themselves.*

In [ ]:
from acm.estimators.compression import IndexedObject, ObjectGroup

obj1 = IndexedObject(indexes={"i": 0, "j": 4}, data=tree)
obj2 = IndexedObject(indexes={"i": 1, "j": 3}, data=tree)
obj3 = IndexedObject(indexes={"i": 2, "j": 2}, data=tree)
example_group = ObjectGroup([obj1, obj2, obj3]) # By construction this object sorts by i, then j
print("Before sorting:", [obj.indexes for obj in example_group.objects])

sorted_group = example_group.sort("j", "i")  # Sorts the IndexedObject instances in the group according to the order of indexes j, then i
print("After sorting:", [obj.indexes for obj in sorted_group.objects])

In the case of the compression, the sorting is handled internally. If the sorting order matches the entire index name list, the dimensions will be ordered accordingly (see the `compress` method).
Otherwise, it is not possible to infer the expected order, so the dimensions will not be reordered.

Once merged and transformed, the `ObjectGroup` can be compressed into a single `xarray.DataArray` object. The `compress` method takes an optional `order` argument for sorting, and an optional `reindex` argument for reindexing some nested indexes (in the case of sparse nested indexes, to avoid NaN values in the final `xarray.DataArray` object). The `reindex` argument takes a dictionary of each index to reindex, and the list of values to reindex to. The `compress` method will return a single `xarray.DataArray` object with the compressed data.

In [ ]:
result = Compressor.compress(
    data=group,
    order=["i", "j", "k"],  # Optional: specify the order of indexes for sorting.
    reindex={"k": ["i", "j"]},  # Optional: specify the reindexing for some nested indexes to avoid NaN values in the final xarray.DataArray object.
    drop_single = True  # Optional: drop single-dimensional coordinates in the final xarray.DataArray object.
)

result

In [ ]:
# Cleanup: delete the mock data directory after the tests
import shutil

shutil.rmtree("mock_data")

**To learn more about this format, check out the Observable documentation.**